In [2]:
from pathlib import Path
import shutil
import pandas as pd

# =========================
# 0) Paths
# =========================
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

v12_dir = ROOT / "data" / "v1_2"
v12_dir.mkdir(parents=True, exist_ok=True)

labels_path = ROOT / "data" / "processed" / "standardized" / "labels_mvp_v1.csv"
qc_path = ROOT / "data" / "manual" / "label_qc_audit_v1.csv"
review_queue_path = ROOT / "data" / "manual" / "label_review_queue_v1.csv"
auto_pass_path = ROOT / "data" / "manual" / "label_auto_pass_sample_v1.csv"

candidate_rule_path = ROOT / "data" / "processed" / "standardized" / "03_candidate_events_rule_pretag_v1.csv"
finbert_path = ROOT / "data" / "processed" / "standardized" / "04_finbert_standardized_all.csv"

modeling_input_out = v12_dir / "stage1_event_level_modeling_input_v1_2.csv"
modeling_summary_out = v12_dir / "stage1_event_level_modeling_summary_v1_2.csv"

# =========================
# 1) Backup current outputs to v1_2
# =========================
for src, dst_name in [
    (labels_path, "labels_mvp_v1.csv"),
    (qc_path, "label_qc_audit_v1.csv"),
    (review_queue_path, "label_review_queue_v1.csv"),
    (auto_pass_path, "label_auto_pass_sample_v1.csv"),
]:
    if src.exists():
        shutil.copy2(src, v12_dir / dst_name)
    else:
        print(f"[WARN] missing backup source: {src}")

# =========================
# 2) Load source tables
# =========================
labels_df = pd.read_csv(labels_path)
cand_df = pd.read_csv(candidate_rule_path)
fin_df = pd.read_csv(finbert_path)

# Keep only required FinBERT fields
fin_keep = [
    "text_id", "source", "term_id",
    "finbert_label", "finbert_pos", "finbert_neu", "finbert_neg",
    "is_trump_direct_text", "is_retweet", "in_market_hours",
]
missing_fin = [c for c in fin_keep if c not in fin_df.columns]
if missing_fin:
    raise ValueError(f"FinBERT table missing columns: {missing_fin}")

fin_df = fin_df[fin_keep].drop_duplicates(subset=["text_id", "source", "term_id"])

# Candidate + FinBERT
base_df = cand_df.merge(fin_df, on=["text_id", "source", "term_id"], how="left")

# =========================
# 3) Merge labels safely (avoid _x/_y collisions)
# =========================
label_keep = [
    "event_id", "label_retreat", "confidence", "review_flag", "review_reason",
    "rule_label", "rule_llm_conflict", "evidence_span", "key_event_id",
    "audit_tier", "model_name",
]
missing_label = [c for c in label_keep if c not in labels_df.columns]
if missing_label:
    raise ValueError(f"Labels table missing columns: {missing_label}")

# Drop overlapping columns from base_df before merge, except join key
overlap_cols = [c for c in label_keep if c != "event_id" and c in base_df.columns]
if overlap_cols:
    base_df = base_df.drop(columns=overlap_cols)

base_df = base_df.merge(labels_df[label_keep], on="event_id", how="left")

# =========================
# 4) Derived columns for notebook 06
# =========================
base_df["candidate_priority_high"] = (
    base_df["candidate_priority"].astype(str).str.lower() == "high"
).astype(int)

base_df["has_evidence_span"] = (
    base_df["evidence_span"].fillna("").astype(str).str.strip().ne("")
).astype(int)

base_df["review_flag_yes"] = (
    base_df["review_flag"].fillna("").astype(str).str.lower() == "yes"
).astype(int)

base_df["is_core_top10"] = (
    base_df["audit_tier"].fillna("").astype(str) == "core_top10"
).astype(int)

base_df["is_extended_backup"] = (
    base_df["audit_tier"].fillna("").astype(str) == "extended_backup"
).astype(int)

# Keep only labeled rows for baseline
model_df = base_df.loc[base_df["label_retreat"].notna()].copy()

# =========================
# 5) Align output schema with v1_1 modeling input
# =========================
ordered_cols = [
    "event_id", "text_id", "source", "term_id", "event_time_utc", "event_time_ny",
    "feature_anchor_date", "target_trade_date",
    "label_retreat", "confidence", "review_flag", "review_reason",
    "rule_label", "rule_llm_conflict",
    "candidate_priority", "candidate_priority_high", "keyword_score",
    "follow_up_count_all_48h", "follow_up_count_relevant_48h", "context_mode",
    "finbert_label", "finbert_pos", "finbert_neu", "finbert_neg",
    "has_evidence_span", "review_flag_yes",
    "key_event_id", "audit_tier", "is_core_top10", "is_extended_backup",
    "theme_hits", "action_hits", "object_hits", "relevance_terms",
    "evidence_span", "rule_text", "url", "model_name",
    "is_trump_direct_text", "is_retweet", "in_market_hours", "source_text",
]

for c in ordered_cols:
    if c not in model_df.columns:
        model_df[c] = pd.NA

model_df = model_df[ordered_cols]
model_df["label_retreat"] = pd.to_numeric(model_df["label_retreat"], errors="coerce").astype(int)

model_df.to_csv(modeling_input_out, index=False)

# =========================
# 6) Build summary
# =========================
summary = {
    "n_rows": float(len(model_df)),
    "n_positive": float((model_df["label_retreat"] == 1).sum()),
    "n_negative": float((model_df["label_retreat"] == 0).sum()),
    "positive_rate": float((model_df["label_retreat"] == 1).mean()) if len(model_df) else 0.0,
    "n_first_term": float((model_df["term_id"] == "first_term").sum()),
    "n_second_term": float((model_df["term_id"] == "second_term").sum()),
    "n_core_top10": float((model_df["audit_tier"] == "core_top10").sum()),
    "n_review_flag_yes": float((model_df["review_flag"].fillna("").str.lower() == "yes").sum()),
}
summary_df = pd.DataFrame({"metric": list(summary.keys()), "value": list(summary.values())})
summary_df.to_csv(modeling_summary_out, index=False)

# =========================
# 7) Quick sanity checks
# =========================
print("v1_2 backup + modeling files generated.")
print("modeling_input rows:", len(model_df))
print("positive rows:", int((model_df["label_retreat"] == 1).sum()))
print("output:", modeling_input_out)
print("output:", modeling_summary_out)
print("\nreview_flag columns in output:", [c for c in model_df.columns if "review_flag" in c])

v1_2 backup + modeling files generated.
modeling_input rows: 1102
positive rows: 26
output: /Users/horace/Coding/ML finance/project/data/v1_2/stage1_event_level_modeling_input_v1_2.csv
output: /Users/horace/Coding/ML finance/project/data/v1_2/stage1_event_level_modeling_summary_v1_2.csv

review_flag columns in output: ['review_flag', 'review_flag_yes']


In [ ]:
from pathlib import Path
import shutil

ROOT = Path.cwd().resolve()
while not (ROOT / "plan_v2.md").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SRC = ROOT / "data" / "v1_2" / "stage1_event_level_modeling_input_v1_2.csv"
DST = ROOT / "data" / "integration" / "stage1_event_level_modeling_input_v1_full.csv"

if not SRC.exists():
    raise FileNotFoundError(f"Missing expected modeling input source: {SRC}")

DST.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(SRC, DST)

print("Copied modeling input to:", DST)


# External Features Pipeline

这个 notebook 负责外部特征准备，包括 Google Trends、黄金价格、市场数据、宏观数据，以及最终的日频特征重建。

各部分可以顺序执行，也可以在已有原始数据时从后续章节继续。

## Google Trends Features

抓取、拼接并构造 Google Trends 的日频可用特征。

# Google Trends Pipeline

This notebook collects and processes Google Trends data for the TACO project.

MVP scope:

- Keywords: `tariff`, `inflation`, `war`
- Geography: United States
- Role: auxiliary public-attention features
- Output: daily trend features that can be merged into Stage 1 and regime tables

Important: Google Trends is not the central project feature. We use a conservative, easy-to-explain pipeline and avoid complex daily stitching.

## Dependency Note

This notebook uses `pytrends` for collection. If the import cell fails, install it once in your environment:

```bash
pip install pytrends
```

If `pytrends` is temporarily blocked or unstable, manually export each keyword from the Google Trends website and place the CSV files under `data/raw/google_trends/`. Then continue from the raw-file loading cells.

In [1]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Project paths
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw" / "google_trends"
PROCESSED_DIR = ROOT / "data" / "processed"

RAW_DAILY_PATH = ROOT / "data" / "raw" / "trends_daily.csv"
FEATURES_DAILY_PATH = PROCESSED_DIR / "trends_features_daily.csv"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Raw Google Trends dir:", RAW_DIR)
print("Processed output:", FEATURES_DAILY_PATH)

Project root: /Users/horace/Coding/ML finance/project/google_trends
Raw Google Trends dir: /Users/horace/Coding/ML finance/project/google_trends/data/raw/google_trends
Processed output: /Users/horace/Coding/ML finance/project/google_trends/data/processed/trends_features_daily.csv


## Configuration

Keep the MVP keyword list small. These are broad public-attention proxies, not labels and not causal variables by themselves.

In [2]:
KEYWORDS = ["tariff", "inflation", "war"]

# Use a window that covers the current project sample.
# Adjust only if the text/event sample expands.
START_DATE = "2016-11-01"
END_DATE = "2025-10-25"

# Google Trends parameters
GEO = "US"
TIMEFRAME = f"{START_DATE} {END_DATE}"

# Prefer overlapping collection windows so Google Trends is more likely to return weekly data.
WINDOW_SPECS = [
    ("w1", "2016-11-01", "2020-12-31"),
    ("w2", "2020-07-01", "2024-06-30"),
    ("w3", "2024-01-01", "2025-10-25"),
]

# Conservative event alignment: event T uses trend features available by T-1 or earlier.
EVENT_ALIGNMENT_LAG_DAYS = 1

# When Google returns weekly data, the date usually identifies the week bucket.
# We treat that weekly value as available only after the week has completed.
WEEKLY_AVAILABILITY_LAG_DAYS = 6

# For very long windows, Google Trends may still return monthly observations.
# In that case we only treat the month's value as available after month-end.
MONTHLY_MIN_MEDIAN_DIFF_DAYS = 27

# Stitching settings for overlapping windows of the same keyword.
STITCH_MIN_OVERLAP_POINTS = 8
RENORMALIZE_STITCHED_TO_100 = True

# Google Trends is an unofficial endpoint and can return HTTP 429.
# Keep retries small; if blocked, use the manual CSV fallback described below.
PYTRENDS_MAX_RETRIES = 2
PYTRENDS_RETRY_SLEEP_SECONDS = 60

print("Keywords:", KEYWORDS)
print("Geo:", GEO)
print("Full timeframe:", TIMEFRAME)
print("Window specs:", WINDOW_SPECS)
print("Retries:", PYTRENDS_MAX_RETRIES, "sleep seconds:", PYTRENDS_RETRY_SLEEP_SECONDS)
print("Monthly threshold (median diff days):", MONTHLY_MIN_MEDIAN_DIFF_DAYS)
print("Min overlap points for stitching:", STITCH_MIN_OVERLAP_POINTS)
print("Renormalize stitched series to 0-100:", RENORMALIZE_STITCHED_TO_100)

Keywords: ['tariff', 'inflation', 'war']
Geo: US
Full timeframe: 2016-11-01 2025-10-25
Window specs: [('w1', '2016-11-01', '2020-12-31'), ('w2', '2020-07-01', '2024-06-30'), ('w3', '2024-01-01', '2025-10-25')]
Retries: 2 sleep seconds: 60
Monthly threshold (median diff days): 27
Min overlap points for stitching: 8
Renormalize stitched series to 0-100: True


## Optional: Collect Raw Data with pytrends

Each keyword is pulled separately. This avoids accidental interpretation that different keywords' 0-100 values are directly comparable.

The preferred MVP collection strategy is to pull overlapping windows for the same keyword, which makes Google Trends more likely to return weekly data instead of monthly data.

Google Trends may return HTTP 429 (`Too Many Requests`). If that happens, this cell records the missing keyword-window files and continues. Manually download the missing CSV files with the same filenames, then continue from the raw-file loading section.

In [4]:
def safe_keyword_name(keyword: str) -> str:
    """Create a stable column/file suffix from a keyword."""
    return keyword.strip().lower().replace(" ", "_").replace("-", "_")


def raw_window_path(keyword: str, window_label: str) -> Path:
    suffix = safe_keyword_name(keyword)
    return RAW_DIR / f"trend_{suffix}_{window_label}_raw.csv"


def collect_one_keyword_window(keyword: str, window_label: str, start_date: str, end_date: str, force: bool = False):
    """Collect one keyword for one time window and save it as a raw CSV."""
    suffix = safe_keyword_name(keyword)
    value_col = f"trend_{suffix}"
    out_path = raw_window_path(keyword, window_label)

    if out_path.exists() and not force:
        print(f"Skip existing raw file: {out_path.name}")
        return out_path

    try:
        from pytrends.request import TrendReq
    except ImportError as exc:
        raise ImportError(
            "pytrends is not installed. Install it with `pip install pytrends`, "
            "or manually export Google Trends CSV files."
        ) from exc

    timeframe = f"{start_date} {end_date}"
    last_error = None
    for attempt in range(1, PYTRENDS_MAX_RETRIES + 1):
        try:
            pytrends = TrendReq(hl="en-US", tz=360, timeout=(10, 30), retries=0, backoff_factor=0)
            pytrends.build_payload([keyword], cat=0, timeframe=timeframe, geo=GEO, gprop="")
            df = pytrends.interest_over_time().reset_index()
            break
        except Exception as exc:
            last_error = exc
            print(
                f"Google Trends request failed for {keyword} [{window_label}] on attempt {attempt}: "
                f"{type(exc).__name__}: {exc}"
            )
            if attempt < PYTRENDS_MAX_RETRIES:
                print(f"Sleeping {PYTRENDS_RETRY_SLEEP_SECONDS} seconds before retry...")
                time.sleep(PYTRENDS_RETRY_SLEEP_SECONDS)
    else:
        print(f"Could not collect {keyword} [{window_label}] with pytrends. Please use manual CSV fallback.")
        print(f"Expected manual file path: {out_path}")
        print(f"Last error: {type(last_error).__name__}: {last_error}")
        return None

    if df.empty:
        print(f"No Google Trends data returned for {keyword} [{window_label}]. Please use manual CSV fallback.")
        print(f"Expected manual file path: {out_path}")
        return None

    df = df.rename(columns={"date": "date", keyword: value_col})
    keep_cols = ["date", value_col]
    if "isPartial" in df.columns:
        keep_cols.append("isPartial")
    df = df[keep_cols]
    df.to_csv(out_path, index=False)

    print(f"Saved: {out_path}")
    return out_path


# Set force=True only if you intentionally want to refresh existing raw files.
raw_paths = []
failed_downloads = []
for keyword in KEYWORDS:
    for window_label, window_start, window_end in WINDOW_SPECS:
        path = collect_one_keyword_window(keyword, window_label, window_start, window_end, force=False)
        if path is None:
            failed_downloads.append((keyword, window_label, window_start, window_end))
        else:
            raw_paths.append(path)
        time.sleep(10)  # be gentle with the unofficial Google Trends endpoint

if failed_downloads:
    print("Manual CSV fallback needed for:")
    for keyword, window_label, window_start, window_end in failed_downloads:
        print(f"- {keyword} [{window_label}] {window_start} to {window_end}")
        print(f"  save as: {raw_window_path(keyword, window_label)}")

print("Collected raw files:", len(raw_paths))
raw_paths[:6]

Skip existing raw file: trend_tariff_w1_raw.csv


KeyboardInterrupt: 

## Load Raw Trend Files

This cell stitches overlapping raw windows into one keyword-level series. It also supports a legacy single raw CSV as a fallback.

In [5]:
def read_csv_with_possible_metadata(csv_path: Path) -> pd.DataFrame:
    """Read pytrends CSV or a manual Google Trends export with metadata rows."""
    df0 = pd.read_csv(csv_path)
    cols0 = [str(c).strip().lower() for c in df0.columns]
    if any(c in {"date", "day", "week", "month"} for c in cols0):
        return df0

    lines = csv_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    header_idx = None
    for i, line in enumerate(lines):
        first = line.split(",")[0].strip().lower()
        if first in {"date", "day", "week", "month"}:
            header_idx = i
            break
    if header_idx is None:
        return df0
    return pd.read_csv(csv_path, skiprows=header_idx)


def read_trend_csv(csv_path: Path, value_col: str) -> pd.DataFrame:
    df = read_csv_with_possible_metadata(csv_path)
    df.columns = [str(c).strip() for c in df.columns]

    date_candidates = [c for c in df.columns if str(c).strip().lower() in {"date", "day", "week", "month"}]
    if "date" not in df.columns:
        if not date_candidates:
            raise ValueError(f"Expected a date-like column in {csv_path.name}")
        df = df.rename(columns={date_candidates[0]: "date"})

    if value_col not in df.columns:
        candidates = [c for c in df.columns if c not in {"date", "isPartial"} and not str(c).lower().startswith("ispartial")]
        if len(candidates) != 1:
            raise ValueError(f"Could not identify trend value column in {csv_path.name}")
        df = df.rename(columns={candidates[0]: value_col})

    out = df[["date", value_col]].copy()
    out["date"] = pd.to_datetime(out["date"])
    out[value_col] = pd.to_numeric(out[value_col], errors="coerce")
    out = out.dropna(subset=["date"]).sort_values("date")
    out = out.drop_duplicates(subset=["date"], keep="last")
    return out


def infer_frequency_kind(dates: pd.Series) -> str:
    """Infer whether a trend series looks daily, weekly, or monthly."""
    diffs = dates.sort_values().diff().dropna().dt.days
    if diffs.empty:
        return "unknown"
    median_diff = diffs.median()
    if median_diff >= MONTHLY_MIN_MEDIAN_DIFF_DAYS:
        return "monthly"
    if median_diff >= 6:
        return "weekly"
    if median_diff <= 2:
        return "daily"
    return "irregular"


def find_keyword_raw_paths(keyword: str) -> list[Path]:
    suffix = safe_keyword_name(keyword)
    window_paths = [raw_window_path(keyword, label) for label, _, _ in WINDOW_SPECS]
    existing_window_paths = [p for p in window_paths if p.exists()]
    if existing_window_paths:
        missing = [p for p in window_paths if not p.exists()]
        if missing:
            missing_str = ", ".join(str(p) for p in missing)
            raise FileNotFoundError(f"Missing window raw files for {keyword}: {missing_str}")
        return existing_window_paths

    legacy_path = RAW_DIR / f"trend_{suffix}_raw.csv"
    if legacy_path.exists():
        return [legacy_path]

    raise FileNotFoundError(f"Missing raw file(s) for {keyword}")


def compute_overlap_scale(left: pd.DataFrame, right: pd.DataFrame, value_col: str) -> tuple[float, int]:
    overlap = left.merge(right, on="date", how="inner", suffixes=("_left", "_right"))
    left_col = f"{value_col}_left"
    right_col = f"{value_col}_right"
    overlap = overlap[(overlap[left_col] > 0) & (overlap[right_col] > 0)]
    if len(overlap) < STITCH_MIN_OVERLAP_POINTS:
        return 1.0, len(overlap)

    left_median = overlap[left_col].median()
    right_median = overlap[right_col].median()
    if pd.isna(left_median) or pd.isna(right_median) or right_median <= 0:
        return 1.0, len(overlap)
    return float(left_median / right_median), len(overlap)


def stitch_keyword_segments(segments: list[pd.DataFrame], value_col: str) -> tuple[pd.DataFrame, list[tuple[int, float, int]]]:
    ordered = sorted(segments, key=lambda x: x["date"].min())
    stitched = ordered[0].copy()
    scale_log = []

    for idx, seg in enumerate(ordered[1:], start=2):
        factor, overlap_n = compute_overlap_scale(stitched, seg, value_col)
        seg_scaled = seg.copy()
        seg_scaled[value_col] = seg_scaled[value_col] * factor
        stitched = pd.concat([stitched, seg_scaled], ignore_index=True)
        stitched = (
            stitched.groupby("date", as_index=False)[value_col]
            .mean()
            .sort_values("date")
        )
        scale_log.append((idx, factor, overlap_n))

    stitched[value_col] = stitched[value_col].clip(lower=0)
    if RENORMALIZE_STITCHED_TO_100 and stitched[value_col].max() > 0:
        stitched[value_col] = 100 * stitched[value_col] / stitched[value_col].max()

    return stitched, scale_log


missing_raw_files = []
raw_series = {}
raw_parts = {}
for keyword in KEYWORDS:
    suffix = safe_keyword_name(keyword)
    value_col = f"trend_{suffix}"
    try:
        paths = find_keyword_raw_paths(keyword)
        segments = [read_trend_csv(path, value_col) for path in paths]
        raw_parts[keyword] = list(zip(paths, segments))
        raw_series[keyword], scale_log = stitch_keyword_segments(segments, value_col)
        print(f"{keyword}: loaded {len(paths)} raw file(s)")
        for path, segment in zip(paths, segments):
            freq_kind = infer_frequency_kind(segment["date"])
            print(
                f"  - {path.name}: {segment['date'].min().date()} to {segment['date'].max().date()} "
                f"rows={len(segment)} freq={freq_kind}"
            )
        for idx, factor, overlap_n in scale_log:
            print(f"  - stitched segment {idx}: scale={factor:.4f}, overlap_points={overlap_n}")
    except FileNotFoundError as exc:
        missing_raw_files.append((keyword, str(exc)))

if missing_raw_files:
    print("Missing Google Trends raw files:")
    for keyword, message in missing_raw_files:
        print(f"- {keyword}: {message}")
    raise RuntimeError("Please collect or manually download the missing Google Trends CSV files, then rerun this cell.")

for keyword, df in raw_series.items():
    freq_kind = infer_frequency_kind(df["date"])
    print(keyword, df["date"].min().date(), "to", df["date"].max().date(), "rows=", len(df), "freq=", freq_kind)
    display(df.head(3))

tariff: loaded 3 raw file(s)
  - trend_tariff_w1_raw.csv: 2016-10-30 to 2020-12-27 rows=218 freq=weekly
  - trend_tariff_w2_raw.csv: 2020-06-28 to 2024-06-30 rows=210 freq=weekly
  - trend_tariff_w3_raw.csv: 2023-12-31 to 2025-10-19 rows=95 freq=weekly
  - stitched segment 2: scale=0.2128, overlap_points=27
  - stitched segment 3: scale=5.3901, overlap_points=27
inflation: loaded 3 raw file(s)
  - trend_inflation_w1_raw.csv: 2016-10-30 to 2020-12-27 rows=218 freq=weekly
  - trend_inflation_w2_raw.csv: 2020-06-28 to 2024-06-30 rows=210 freq=weekly
  - trend_inflation_w3_raw.csv: 2023-12-31 to 2025-10-19 rows=95 freq=weekly
  - stitched segment 2: scale=5.0000, overlap_points=27
  - stitched segment 3: scale=2.6667, overlap_points=27
war: loaded 3 raw file(s)
  - trend_war_w1_raw.csv: 2016-10-30 to 2020-12-27 rows=218 freq=weekly
  - trend_war_w2_raw.csv: 2020-06-28 to 2024-06-30 rows=210 freq=weekly
  - trend_war_w3_raw.csv: 2023-12-31 to 2025-10-19 rows=95 freq=weekly
  - stitched segm

,date,trend_tariff
0,2016-10-30,3.525000
1,2016-11-06,3.896053
2,2016-11-13,4.267105


inflation 2016-10-30 to 2025-10-19 rows= 469 freq= weekly


,date,trend_inflation
0,2016-10-30,14.0
1,2016-11-06,13.8
2,2016-11-13,15.6


war 2016-10-30 to 2025-10-19 rows= 469 freq= weekly


,date,trend_war
0,2016-10-30,26.0
1,2016-11-06,26.0
2,2016-11-13,26.0


## Convert to Daily Available Series

The preferred raw input is a stitched weekly series built from overlapping windows. To avoid look-ahead bias, we treat each low-frequency bucket as available only after that bucket completes, then forward-fill it to daily frequency.

In [6]:
def to_daily_available(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    """Convert raw trend observations into conservative daily available values."""
    out = df[["date", value_col]].copy().sort_values("date")
    freq_kind = infer_frequency_kind(out["date"])

    if freq_kind == "monthly":
        # A full-month value should only be treated as known after that month ends.
        out["available_date"] = out["date"] + pd.offsets.MonthEnd(1)
    elif freq_kind == "weekly":
        # Conservative rule: the week's full value is only available after the week ends.
        out["available_date"] = out["date"] + pd.Timedelta(days=WEEKLY_AVAILABILITY_LAG_DAYS)
    else:
        # Daily or irregular data is still lagged by one day before event-level use.
        out["available_date"] = out["date"]

    daily = (
        out[["available_date", value_col]]
        .rename(columns={"available_date": "date"})
        .drop_duplicates(subset=["date"], keep="last")
        .set_index("date")
        .sort_index()
    )

    full_index = pd.date_range(pd.to_datetime(START_DATE), pd.to_datetime(END_DATE), freq="D")
    daily = daily.reindex(full_index).ffill().rename_axis("date").reset_index()
    print(f"{value_col}: detected {freq_kind}, daily rows={len(daily)}")
    return daily


daily_frames = []
for keyword, df in raw_series.items():
    suffix = safe_keyword_name(keyword)
    value_col = f"trend_{suffix}"
    daily_frames.append(to_daily_available(df, value_col))

trends_daily = daily_frames[0]
for frame in daily_frames[1:]:
    trends_daily = trends_daily.merge(frame, on="date", how="outer")

trends_daily = trends_daily.sort_values("date").reset_index(drop=True)
trends_daily.to_csv(RAW_DAILY_PATH, index=False)

print("Saved daily raw trends:", RAW_DAILY_PATH)
display(trends_daily.head())
display(trends_daily.tail())

trend_tariff: detected weekly, daily rows=3281
trend_inflation: detected weekly, daily rows=3281
trend_war: detected weekly, daily rows=3281
Saved daily raw trends: /Users/horace/Coding/ML finance/project/google_trends/data/raw/trends_daily.csv


,date,trend_tariff,trend_inflation,trend_war
0,2016-11-01,NaN,NaN,NaN
1,2016-11-02,NaN,NaN,NaN
2,2016-11-03,NaN,NaN,NaN
3,2016-11-04,NaN,NaN,NaN
4,2016-11-05,3.525,14.0,26.0


,date,trend_tariff,trend_inflation,trend_war
3276,2025-10-21,13.0,30.933333,25.190476
3277,2025-10-22,13.0,30.933333,25.190476
3278,2025-10-23,13.0,30.933333,25.190476
3279,2025-10-24,13.0,30.933333,25.190476
3280,2025-10-25,12.0,36.266667,23.422723


## Build Rolling Features

The most useful MVP variables are rolling z-scores. They say whether a topic's attention is unusually high relative to its own recent history.

In [7]:
def add_trend_features(df: pd.DataFrame, base_col: str) -> pd.DataFrame:
    """Add simple rolling features for one Google Trends column."""
    out = df.copy()
    s = out[base_col].astype(float)

    out[f"{base_col}_ma_7d"] = s.rolling(7, min_periods=3).mean()
    out[f"{base_col}_ma_20d"] = s.rolling(20, min_periods=7).mean()

    rolling_mean = s.rolling(60, min_periods=20).mean()
    rolling_std = s.rolling(60, min_periods=20).std()
    z_col = f"{base_col}_z_60d"
    out[z_col] = (s - rolling_mean) / rolling_std.replace(0, np.nan)
    out[f"{base_col}_spike"] = (out[z_col] >= 2).astype("Int64")
    return out


trend_cols = [f"trend_{safe_keyword_name(k)}" for k in KEYWORDS]
features_daily = trends_daily.copy()

for col in trend_cols:
    features_daily = add_trend_features(features_daily, col)

features_daily.to_csv(FEATURES_DAILY_PATH, index=False)

print("Saved feature table:", FEATURES_DAILY_PATH)
print("Rows:", len(features_daily))
display(features_daily.head())
display(features_daily.tail())

Saved feature table: /Users/horace/Coding/ML finance/project/google_trends/data/processed/trends_features_daily.csv
Rows: 3281


,date,trend_tariff,trend_inflation,trend_war,trend_tariff_ma_7d,trend_tariff_ma_20d,trend_tariff_z_60d,trend_tariff_spike,trend_inflation_ma_7d,trend_inflation_ma_20d,trend_inflation_z_60d,trend_inflation_spike,trend_war_ma_7d,trend_war_ma_20d,trend_war_z_60d,trend_war_spike
0,2016-11-01,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,0
1,2016-11-02,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,0
2,2016-11-03,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,0
3,2016-11-04,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,0
4,2016-11-05,3.525,14.0,26.0,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,0


,date,trend_tariff,trend_inflation,trend_war,trend_tariff_ma_7d,trend_tariff_ma_20d,trend_tariff_z_60d,trend_tariff_spike,trend_inflation_ma_7d,trend_inflation_ma_20d,trend_inflation_z_60d,trend_inflation_spike,trend_war_ma_7d,trend_war_ma_20d,trend_war_z_60d,trend_war_spike
3276,2025-10-21,13.0,30.933333,25.190476,12.571429,11.75,1.385029,0,31.847619,33.733333,0.173222,0,25.001074,25.190476,0.894644,0
3277,2025-10-22,13.0,30.933333,25.190476,12.714286,11.85,1.335025,0,31.542857,33.546667,0.148731,0,25.064208,25.190476,0.866496,0
3278,2025-10-23,13.0,30.933333,25.190476,12.857143,11.95,1.287811,0,31.238095,33.360000,0.123453,0,25.127342,25.190476,0.839073,0
3279,2025-10-24,13.0,30.933333,25.190476,13.000000,12.05,1.243076,0,30.933333,33.120000,0.097253,0,25.190476,25.168379,0.812312,0
3280,2025-10-25,12.0,36.266667,23.422723,12.857143,12.10,0.543990,0,31.695238,33.146667,1.619965,0,24.937940,25.057895,-2.129685,0


## Quality Checks

These checks are intentionally simple. The goal is to catch obvious problems before merging with events.

In [8]:
def quality_report(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        rows.append({
            "column": col,
            "missing_count": int(df[col].isna().sum()),
            "missing_rate": float(df[col].isna().mean()),
            "dtype": str(df[col].dtype),
        })
    return pd.DataFrame(rows)


print("Date range:", features_daily["date"].min(), "to", features_daily["date"].max())
print("Duplicate dates:", features_daily["date"].duplicated().sum())
display(quality_report(features_daily))

required_cols = [
    "trend_tariff_z_60d",
    "trend_inflation_z_60d",
    "trend_war_z_60d",
]
missing_required = [c for c in required_cols if c not in features_daily.columns]
if missing_required:
    raise ValueError(f"Missing required MVP feature columns: {missing_required}")

print("Required MVP feature columns are present.")

Date range: 2016-11-01 00:00:00 to 2025-10-25 00:00:00
Duplicate dates: 0


,column,missing_count,missing_rate,dtype
0,date,0,0.000000,datetime64[us]
1,trend_tariff,4,0.001219,float64
2,trend_inflation,4,0.001219,float64
3,trend_war,4,0.001219,float64
4,trend_tariff_ma_7d,6,0.001829,float64
5,trend_tariff_ma_20d,10,0.003048,float64
6,trend_tariff_z_60d,23,0.007010,float64
7,trend_tariff_spike,0,0.000000,Int64
8,trend_inflation_ma_7d,6,0.001829,float64
9,trend_inflation_ma_20d,10,0.003048,float64


Required MVP feature columns are present.


## Example: Align Trends to Event-Level Table

This is an example only. It does not overwrite the project event table.

Rule: an event on date `T` uses Google Trends features available by `T - 1 day` or earlier.

In [9]:
EVENTS_PATH = ROOT / "data" / "processed" / "standardized" / "03_candidate_events_standardized.csv"

if EVENTS_PATH.exists():
    events = pd.read_csv(EVENTS_PATH)
    print("Loaded events:", EVENTS_PATH)
    print("Event columns sample:", list(events.columns[:12]))

    # Try common event-time columns used in this project.
    event_time_col = None
    for candidate in ["event_time_ny", "published_at_ny", "event_time_utc", "published_at_utc"]:
        if candidate in events.columns:
            event_time_col = candidate
            break

    if event_time_col is None:
        print("No known event timestamp column found. Please set event_time_col manually.")
    else:
        events_example = events.copy()
        events_example["event_date"] = pd.to_datetime(events_example[event_time_col], errors="coerce").dt.normalize()
        events_example["trend_lookup_date"] = events_example["event_date"] - pd.Timedelta(days=EVENT_ALIGNMENT_LAG_DAYS)

        trends_for_merge = features_daily.copy()
        trends_for_merge["date"] = pd.to_datetime(trends_for_merge["date"])

        merged_example = pd.merge_asof(
            events_example.sort_values("trend_lookup_date"),
            trends_for_merge.sort_values("date"),
            left_on="trend_lookup_date",
            right_on="date",
            direction="backward",
        )

        print("Merged example shape:", merged_example.shape)
        display(merged_example[["event_date", "trend_lookup_date", "date"] + required_cols].head())
else:
    print("Event file not found. Skip event alignment example:", EVENTS_PATH)

Event file not found. Skip event alignment example: /Users/horace/Coding/ML finance/project/google_trends/data/processed/standardized/03_candidate_events_standardized.csv


## Recommended Columns for Modeling

For the MVP model, start with:

```text
trend_tariff_z_60d
trend_inflation_z_60d
trend_war_z_60d
```

Use raw levels and moving averages only for robustness checks or diagnostics.

## Gold Price Data

下载或导入黄金日频价格数据。

# Gold Price Download Only

这个 notebook 只负责下载黄金日频价格，优先使用 Alpha Vantage 的黄金历史接口；如果在线下载失败，也支持从本地 CSV 导入。

输出文件：`data/raw/market_yf_daily.csv`

输出列：`date`, `gold_close`, `gold_source`


## Dependency Note

如果环境里缺依赖，先运行：

```python
pip install requests pandas
```

如果你要用 Alpha Vantage 在线下载，还需要先申请一个免费 API key：
`https://www.alphavantage.co/support/#api-key`


In [7]:
from pathlib import Path
import io
import os
import warnings

import pandas as pd
import requests
from IPython.display import display

warnings.filterwarnings("ignore")

ROOT = Path.cwd().resolve()
if ROOT.name == "market_macro":
    ROOT = ROOT.parent
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"
MARKET_RAW_PATH = RAW_DIR / "market_yf_daily.csv"

RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Gold raw output:", MARKET_RAW_PATH)


Project root: /Users/horace/Coding/ML finance/project
Gold raw output: /Users/horace/Coding/ML finance/project/data/raw/market_yf_daily.csv


## Configuration

两种方式任选其一：

1. 在线下载：填写 `ALPHAVANTAGE_API_KEY`
2. 本地导入：填写 `LOCAL_GOLD_CSV_PATH`


In [10]:
START_DATE = "2016-11-01"
END_DATE = "2025-10-25"

# Option A: Alpha Vantage online download
ALPHAVANTAGE_API_KEY = os.getenv("ALPHAVANTAGE_API_KEY", "7RRHA4PPB2XH1O3L")
AV_SYMBOL = "GOLD"
AV_INTERVAL = "daily"

# Option B: local CSV fallback
# Example: ROOT / "data" / "raw" / "gold_manual.csv"
LOCAL_GOLD_CSV_PATH = ""

print("Date window:", START_DATE, "to", END_DATE)
print("Alpha Vantage symbol:", AV_SYMBOL)
print("Alpha Vantage interval:", AV_INTERVAL)
print("Alpha Vantage key configured:", bool(ALPHAVANTAGE_API_KEY))
print("Local CSV path:", LOCAL_GOLD_CSV_PATH or "<not set>")


Date window: 2016-11-01 to 2025-10-25
Alpha Vantage symbol: GOLD
Alpha Vantage interval: daily
Alpha Vantage key configured: True
Local CSV path: <not set>


## Collect Gold Data

默认逻辑：

1. 如果 `data/raw/market_yf_daily.csv` 已存在且 `force=False`，直接复用
2. 如果配置了 Alpha Vantage key，优先在线下载
3. 如果在线下载失败，再尝试本地 CSV


In [11]:
def normalize_gold_frame(raw: pd.DataFrame, source: str) -> pd.DataFrame:
    date_candidates = ["date", "Date", "timestamp", "Timestamp", "time"]
    value_candidates = [
        "gold_close",
        "close",
        "Close",
        "value",
        "Value",
        "price",
        "Price",
    ]

    date_col = next((c for c in date_candidates if c in raw.columns), None)
    value_col = next((c for c in value_candidates if c in raw.columns), None)

    if date_col is None or value_col is None:
        raise ValueError(
            f"Could not identify date/value columns for {source}. Columns: {list(raw.columns)}"
        )

    out = raw[[date_col, value_col]].copy()
    out = out.rename(columns={date_col: "date", value_col: "gold_close"})
    out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.normalize()
    out["gold_close"] = pd.to_numeric(out["gold_close"], errors="coerce")
    out = out.dropna(subset=["date", "gold_close"])
    out = out[(out["date"] >= pd.to_datetime(START_DATE)) & (out["date"] <= pd.to_datetime(END_DATE))]
    out = out.sort_values("date").drop_duplicates(subset=["date"], keep="last")
    return out.reset_index(drop=True)


def download_alpha_vantage_gold(api_key: str) -> tuple[pd.DataFrame | None, str | None]:
    if not api_key:
        err = "Alpha Vantage API key is empty."
        print(err)
        return None, err

    url = "https://www.alphavantage.co/query"
    params = {
        "function": "GOLD_SILVER_HISTORY",
        "symbol": AV_SYMBOL,
        "interval": AV_INTERVAL,
        "datatype": "csv",
        "apikey": api_key,
    }

    try:
        resp = requests.get(url, params=params, timeout=90)
        resp.raise_for_status()
    except Exception as exc:
        err = f"Alpha Vantage request failed: {type(exc).__name__}: {exc}"
        print(err)
        return None, err

    text = resp.text.strip()
    if not text:
        err = "Alpha Vantage returned empty response."
        print(err)
        return None, err

    lowered = text.lower()
    if lowered.startswith("{"):
        err = f"Alpha Vantage returned JSON instead of CSV: {text[:300]}"
        print(err)
        return None, err

    if "Error Message" in text or "Information" in text or "Note" in text:
        err = f"Alpha Vantage returned API message: {text[:300]}"
        print(err)
        return None, err

    try:
        raw = pd.read_csv(io.StringIO(text))
        out = normalize_gold_frame(raw, "Alpha Vantage")
    except Exception as exc:
        err = f"Alpha Vantage parse failed: {type(exc).__name__}: {exc}"
        print(err)
        return None, err

    if out.empty:
        err = f"Alpha Vantage returned no rows inside date window {START_DATE} to {END_DATE}."
        print(err)
        return None, err

    return out, None


def load_local_gold_csv(path_str: str) -> tuple[pd.DataFrame | None, str | None]:
    if not path_str:
        err = "Local CSV path is empty."
        print(err)
        return None, err

    path = Path(path_str)
    if not path.is_absolute():
        path = ROOT / path

    if not path.exists():
        err = f"Local CSV does not exist: {path}"
        print(err)
        return None, err

    try:
        raw = pd.read_csv(path)
        out = normalize_gold_frame(raw, f"local CSV {path}")
    except Exception as exc:
        err = f"Local CSV parse failed: {type(exc).__name__}: {exc}"
        print(err)
        return None, err

    if out.empty:
        err = f"Local CSV returned no rows inside date window {START_DATE} to {END_DATE}."
        print(err)
        return None, err

    return out, None


def collect_gold_data(force: bool = False) -> pd.DataFrame:
    if MARKET_RAW_PATH.exists() and not force:
        print("Using existing gold raw file:", MARKET_RAW_PATH)
        return pd.read_csv(MARKET_RAW_PATH, parse_dates=["date"])

    gold = None
    source = None
    av_err = None
    local_err = None

    if ALPHAVANTAGE_API_KEY:
        print("Downloading gold from Alpha Vantage...")
        gold, av_err = download_alpha_vantage_gold(ALPHAVANTAGE_API_KEY)
        if gold is not None and not gold.empty:
            source = f"AlphaVantage:{AV_SYMBOL}:{AV_INTERVAL}"
    else:
        print("Alpha Vantage key not set; skipping online download.")

    if gold is None and LOCAL_GOLD_CSV_PATH:
        print("Trying local gold CSV fallback...")
        gold, local_err = load_local_gold_csv(LOCAL_GOLD_CSV_PATH)
        if gold is not None and not gold.empty:
            source = "LOCAL_CSV"

    if gold is None or gold.empty:
        raise RuntimeError(
            "Gold download failed.\n"
            f"Alpha Vantage detail: {av_err}\n"
            f"Local CSV detail: {local_err}\n"
            "You can either configure ALPHAVANTAGE_API_KEY or set LOCAL_GOLD_CSV_PATH to a CSV with columns like date,gold_close."
        )

    gold["gold_source"] = source
    gold.to_csv(MARKET_RAW_PATH, index=False)

    print("Saved:", MARKET_RAW_PATH)
    print("Source log:", source)
    return gold


# Set force=True only if you intentionally want to refresh the raw file.
gold_raw = collect_gold_data(force=False)
display(gold_raw.head())
display(gold_raw.tail())


Saved: /Users/horace/Coding/ML finance/project/data/raw/market_yf_daily.csv
Source log: AlphaVantage:GOLD:daily


,date,gold_close,gold_source
0,2016-11-01,1302.519715,AlphaVantage:GOLD:daily
1,2016-11-02,1308.589123,AlphaVantage:GOLD:daily
2,2016-11-03,1311.193116,AlphaVantage:GOLD:daily
3,2016-11-04,1311.596841,AlphaVantage:GOLD:daily
4,2016-11-05,1315.555789,AlphaVantage:GOLD:daily


,date,gold_close,gold_source
3276,2025-10-21,4359.148501,AlphaVantage:GOLD:daily
3277,2025-10-22,4010.895196,AlphaVantage:GOLD:daily
3278,2025-10-23,4088.803914,AlphaVantage:GOLD:daily
3279,2025-10-24,4130.628653,AlphaVantage:GOLD:daily
3280,2025-10-25,4113.311870,AlphaVantage:GOLD:daily


## Local CSV Template

如果你走本地导入，最简单的 CSV 格式是：

```csv
date,gold_close
2016-11-01,1277.15
2016-11-02,1288.00
```

也可以接受常见列名，比如 `timestamp, close` 或 `Date, Close`。


## Optional Refresh

如果你已经生成过 `data/raw/market_yf_daily.csv`，默认会直接复用已有文件。

如果你想强制重新下载或重新导入，把上一段最后一行改成：

```python
gold_raw = collect_gold_data(force=True)
```


## Market and Macro Data

收集市场与宏观时间序列，并完成基础特征工程与质量检查。

# Market and Macro Data Pipeline

This notebook collects and processes market and macro data for the TACO project.

MVP scope:

- Yahoo Finance: S&P 500, VIX, gold futures, dollar index
- FRED: 10-year TIPS real yield
- Output: daily feature table for Stage 1, GMM regime, and Stage 2

Important: this notebook only prepares data. It does not train models.

## Dependency Note

This notebook uses `yfinance` and `pandas_datareader`.

Install them once if needed:

```bash
pip install yfinance pandas_datareader
```

If online collection fails, manually download the CSV files and continue from the raw-file loading cells.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Project paths
ROOT = Path.cwd().resolve()
if ROOT.name == "market_macro":
    ROOT = ROOT.parent
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"

MARKET_RAW_PATH = RAW_DIR / "market_yf_daily.csv"
MACRO_RAW_PATH = RAW_DIR / "macro_fred_daily.csv"
FEATURES_PATH = PROCESSED_DIR / "market_macro_features_daily.csv"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Market raw output:", MARKET_RAW_PATH)
print("Macro raw output:", MACRO_RAW_PATH)
print("Feature output:", FEATURES_PATH)

## Configuration

The project window starts in 2017, but we collect from late 2016 to support rolling features.

In [ ]:
START_DATE = "2016-11-01"
END_DATE = "2025-10-25"

YF_TICKERS = {
    "spx": "^GSPC",
    "vix": "^VIX",
    "gold": "GC=F",
    "dxy": "DX-Y.NYB",
}

FRED_SERIES = {
    "real_yield_10y": "DFII10",
}

# FRED fallback series used when Yahoo Finance is rate-limited.
# These are close proxies for the same market variables and keep the MVP moving.
FRED_MARKET_FALLBACK = {
    "spx": "SP500",
    "vix": "VIXCLS",
    "gold": "GOLDAMGBD228NLBM",
    "dxy": "DTWEXBGS",
}

print("Date window:", START_DATE, "to", END_DATE)
print("Yahoo tickers:", YF_TICKERS)
print("FRED macro series:", FRED_SERIES)
print("FRED market fallback series:", FRED_MARKET_FALLBACK)

## Collect Market Data

The notebook first tries Yahoo Finance. If Yahoo is rate-limited, it automatically falls back to FRED proxy series so the pipeline can still run.

In [ ]:
def read_fred_series(series_id: str, value_col: str) -> pd.DataFrame | None:
    """Read one FRED series and return standardized date/value columns.

    Returns None if FRED is unavailable or the series cannot be read.
    """
    try:
        from pandas_datareader import data as pdr
    except ImportError as exc:
        raise ImportError(
            "pandas_datareader is required for the FRED fallback. "
            "Install it with `pip install pandas_datareader`."
        ) from exc

    try:
        raw = pdr.DataReader(series_id, "fred", START_DATE, END_DATE).reset_index()
    except Exception as exc:
        print(f"FRED failed for {series_id}: {type(exc).__name__}: {exc}")
        return None

    raw = raw.rename(columns={"DATE": "date", series_id: value_col})
    raw["date"] = pd.to_datetime(raw["date"]).dt.normalize()
    raw[value_col] = pd.to_numeric(raw[value_col], errors="coerce")
    return raw[["date", value_col]].dropna(subset=["date"])


def try_yahoo_one_ticker(name: str, ticker: str) -> pd.DataFrame | None:
    """Try one Yahoo ticker. Return None if Yahoo is rate-limited or empty."""
    try:
        import yfinance as yf
    except ImportError:
        print("yfinance is not installed; will use FRED fallback.")
        return None

    try:
        raw = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False, auto_adjust=False, threads=False)
    except Exception as exc:
        print(f"Yahoo failed for {ticker}: {type(exc).__name__}: {exc}")
        return None

    if raw.empty:
        print(f"Yahoo returned no rows for {ticker}; will use FRED fallback.")
        return None

    raw = raw.reset_index()

    # yfinance can return MultiIndex columns in some versions; flatten them safely.
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = [c[0] if isinstance(c, tuple) else c for c in raw.columns]

    date_col = "Date" if "Date" in raw.columns else "Datetime"
    price_col = "Adj Close" if "Adj Close" in raw.columns else "Close"

    one = raw[[date_col, price_col]].copy()
    one = one.rename(columns={date_col: "date", price_col: f"{name}_close"})
    one["date"] = pd.to_datetime(one["date"]).dt.normalize()
    one[f"{name}_close"] = pd.to_numeric(one[f"{name}_close"], errors="coerce")
    return one[["date", f"{name}_close"]]


def collect_market_data(force: bool = False) -> pd.DataFrame:
    """Download market data and save a clean daily close table.

    Primary source is Yahoo Finance. If a Yahoo ticker is rate-limited or empty,
    the function automatically uses a FRED proxy series for that variable.
    """
    if MARKET_RAW_PATH.exists() and not force:
        print("Using existing market raw file:", MARKET_RAW_PATH)
        return pd.read_csv(MARKET_RAW_PATH, parse_dates=["date"])

    frames = []
    source_log = {}
    missing_market_vars = []
    for name, ticker in YF_TICKERS.items():
        print(f"Downloading {name}: {ticker}")
        one = try_yahoo_one_ticker(name, ticker)
        if one is None:
            fallback_id = FRED_MARKET_FALLBACK[name]
            print(f"Using FRED fallback for {name}: {fallback_id}")
            one = read_fred_series(fallback_id, f"{name}_close")
            if one is None or one.empty:
                print(f"Missing {name}_close after Yahoo and FRED attempts. Manual CSV fill is needed.")
                missing_market_vars.append(name)
                source_log[name] = "MISSING_MANUAL_REQUIRED"
                continue
            else:
                source_log[name] = f"FRED:{fallback_id}"
        else:
            source_log[name] = f"Yahoo:{ticker}"
        frames.append(one)

    if frames:
        market = frames[0]
        for frame in frames[1:]:
            market = market.merge(frame, on="date", how="outer")
    else:
        # Create a placeholder calendar so the notebook can still show exactly what is missing.
        market = pd.DataFrame({"date": pd.date_range(START_DATE, END_DATE, freq="B")})

    for name in YF_TICKERS:
        col = f"{name}_close"
        if col not in market.columns:
            market[col] = np.nan

    market = market.sort_values("date").drop_duplicates(subset=["date"], keep="last")
    for name, source in source_log.items():
        market[f"{name}_source"] = source
    market.to_csv(MARKET_RAW_PATH, index=False)
    print("Saved:", MARKET_RAW_PATH)
    print("Source log:", source_log)
    if missing_market_vars:
        print("Manual fill needed for:", missing_market_vars)
        print("Recommended manual file format: data/raw/market_yf_daily.csv with columns date,spx_close,vix_close,gold_close,dxy_close")
        print("After manually filling the missing columns, rerun from the raw-file loading section.")
    return market


# Set force=True only if you intentionally want to refresh the raw file.
market_raw = collect_market_data(force=False)
display(market_raw.head())
display(market_raw.tail())

## Collect FRED Macro Data

`DFII10` is the 10-year TIPS real yield. It is a key macro control in the project plan.

In [ ]:
def collect_fred_macro_data(force: bool = False) -> pd.DataFrame:
    """Download FRED macro series and save a clean daily table."""
    if MACRO_RAW_PATH.exists() and not force:
        print("Using existing macro raw file:", MACRO_RAW_PATH)
        return pd.read_csv(MACRO_RAW_PATH, parse_dates=["date"])

    frames = []
    missing_macro_vars = []
    for name, series_id in FRED_SERIES.items():
        print(f"Downloading {name}: {series_id}")
        one = read_fred_series(series_id, name)
        if one is None or one.empty:
            print(f"Missing {name} after FRED attempt. Manual CSV fill is needed.")
            missing_macro_vars.append(name)
            continue
        frames.append(one)

    if frames:
        macro = frames[0]
        for frame in frames[1:]:
            macro = macro.merge(frame, on="date", how="outer")
    else:
        macro = pd.DataFrame({"date": pd.date_range(START_DATE, END_DATE, freq="B")})

    for name in FRED_SERIES:
        if name not in macro.columns:
            macro[name] = np.nan

    macro = macro.sort_values("date").drop_duplicates(subset=["date"], keep="last")
    macro.to_csv(MACRO_RAW_PATH, index=False)
    print("Saved:", MACRO_RAW_PATH)
    if missing_macro_vars:
        print("Manual fill needed for:", missing_macro_vars)
        print("Recommended manual file format: data/raw/macro_fred_daily.csv with columns date,real_yield_10y")
        print("After manually filling the missing columns, rerun from the raw-file loading section.")
    return macro


# Set force=True only if you intentionally want to refresh the raw file.
macro_raw = collect_fred_macro_data(force=False)
display(macro_raw.head())
display(macro_raw.tail())

## Manual CSV Fallback Format

If online collection fails, manually create these files with the same columns:

`data/raw/market_yf_daily.csv`

```text
date,spx_close,vix_close,gold_close,dxy_close
```

`data/raw/macro_fred_daily.csv`

```text
date,real_yield_10y
```

Then restart the notebook from the next section.

## Load Raw Files and Merge

FRED macro values are forward-filled after merging to the market calendar. We do not backfill into the past.

In [ ]:
market = pd.read_csv(MARKET_RAW_PATH, parse_dates=["date"])
macro = pd.read_csv(MACRO_RAW_PATH, parse_dates=["date"])

market = market.sort_values("date").drop_duplicates(subset=["date"], keep="last")
macro = macro.sort_values("date").drop_duplicates(subset=["date"], keep="last")

# Use the market calendar as the modeling calendar.
daily = market.merge(macro, on="date", how="left")

# FRED series may be missing on market days; forward-fill only after the first observed value.
macro_cols = list(FRED_SERIES.keys())
daily[macro_cols] = daily[macro_cols].ffill()

daily = daily.sort_values("date").reset_index(drop=True)

print("Merged daily shape:", daily.shape)
display(daily.head())
display(daily.tail())

## Feature Engineering

Keep features simple and transparent. Rolling features use past and current rows in the daily market calendar; event alignment later uses only prior available observations.

In [ ]:
def log_return(series: pd.Series, periods: int = 1) -> pd.Series:
    """Compute log return over a given number of periods."""
    return np.log(series / series.shift(periods))


features = daily.copy()

# One-day log returns
features["spx_ret_1d"] = log_return(features["spx_close"])
features["gold_ret_1d"] = log_return(features["gold_close"])
features["dxy_ret_1d"] = log_return(features["dxy_close"])

# VIX and real-yield changes are differences, not log returns.
features["vix_change_1d"] = features["vix_close"].diff(1)
features["vix_change_5d"] = features["vix_close"].diff(5)
features["real_yield_change_1d"] = features["real_yield_10y"].diff(1)
features["real_yield_change_5d"] = features["real_yield_10y"].diff(5)

# Downside pressure features for Stage 1.
features["spx_drawdown_1d"] = features["spx_ret_1d"].clip(upper=0)
features["spx_drawdown_5d"] = log_return(features["spx_close"], periods=5).clip(upper=0)

# Regime-support features.
features["vix_ma_5d"] = features["vix_close"].rolling(5, min_periods=3).mean()
features["gold_spx_corr_20d"] = features["gold_ret_1d"].rolling(20, min_periods=10).corr(features["spx_ret_1d"])

features.to_csv(FEATURES_PATH, index=False)

print("Saved feature table:", FEATURES_PATH)
print("Rows:", len(features))
display(features.head())
display(features.tail())

## Quality Checks

These checks catch common data problems before the table is used in modeling.

In [ ]:
def quality_report(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        rows.append({
            "column": col,
            "missing_count": int(df[col].isna().sum()),
            "missing_rate": float(df[col].isna().mean()),
            "dtype": str(df[col].dtype),
        })
    return pd.DataFrame(rows)


print("Date range:", features["date"].min().date(), "to", features["date"].max().date())
print("Duplicate dates:", features["date"].duplicated().sum())

core_cols = ["spx_close", "vix_close", "gold_close", "dxy_close", "real_yield_10y"]
missing_core = features[core_cols].isna().mean().sort_values(ascending=False)
print("Core missing rates:")
display(missing_core)

corr = features["gold_spx_corr_20d"].dropna()
if not corr.empty:
    print("Rolling corr min/max:", corr.min(), corr.max())
    assert corr.between(-1.000001, 1.000001).all(), "Rolling correlation outside [-1, 1]"

display(quality_report(features))

## Example: Align Market/Macro Features to Events

This is an example only. It does not overwrite the existing event table.

Rule: an event gets the most recent available market/macro observation before the event.

In [ ]:
EVENTS_PATH = ROOT / "data" / "processed" / "standardized" / "03_candidate_events_standardized.csv"

if EVENTS_PATH.exists():
    events = pd.read_csv(EVENTS_PATH)
    print("Loaded events:", EVENTS_PATH)
    print("Event columns sample:", list(events.columns[:12]))

    event_time_col = None
    for candidate in ["event_time_ny", "published_at_ny", "event_time_utc", "published_at_utc"]:
        if candidate in events.columns:
            event_time_col = candidate
            break

    if event_time_col is None:
        print("No known event timestamp column found. Please set event_time_col manually.")
    else:
        events_example = events.copy()
        events_example["event_time_for_merge"] = pd.to_datetime(events_example[event_time_col], errors="coerce")
        events_example["event_date"] = events_example["event_time_for_merge"].dt.normalize()

        feature_cols = [
            "date",
            "spx_ret_1d",
            "spx_drawdown_1d",
            "spx_drawdown_5d",
            "vix_close",
            "vix_change_1d",
            "gold_spx_corr_20d",
            "dxy_close",
            "real_yield_10y",
        ]

        market_for_merge = features[feature_cols].copy()
        market_for_merge["date"] = pd.to_datetime(market_for_merge["date"])

        merged_example = pd.merge_asof(
            events_example.sort_values("event_date"),
            market_for_merge.sort_values("date"),
            left_on="event_date",
            right_on="date",
            direction="backward",
        )

        print("Merged example shape:", merged_example.shape)
        display(merged_example[["event_date", "date", "spx_drawdown_1d", "vix_change_1d", "real_yield_10y"]].head())
else:
    print("Event file not found. Skip event alignment example:", EVENTS_PATH)

## Recommended MVP Columns

For Stage 1 pressure features, start with:

```text
spx_drawdown_1d
spx_drawdown_5d
vix_change_1d
vix_close
```

For GMM regime features, start with:

```text
vix_close
gold_spx_corr_20d
real_yield_10y
```

For Stage 2 controls and target, start with:

```text
gold_ret_1d
dxy_close
real_yield_10y
```

## Rebuild Daily Market/Macro Features

将黄金序列并入市场原始表，并重建最终日频特征表。

# Rebuild Market Macro Features

作用：
- 合并 `market_yf_daily.csv`、`macro_fred_daily.csv` 与黄金 raw 数据
- 重新生成 `market_macro_features_daily.csv`
- 输出简单质量检查

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'market_macro':
    ROOT = ROOT.parent

MARKET_RAW_PATH = ROOT / 'data' / 'macro_trend' / 'data' / 'raw' / 'market_yf_daily.csv'
MACRO_RAW_PATH = ROOT / 'data' / 'macro_trend' / 'data' / 'raw' / 'macro_fred_daily.csv'
GOLD_RAW_PATH = ROOT / 'data' / 'gold_raw' / 'market_yf_daily.csv'
FEATURES_PATH = ROOT / 'data' / 'macro_trend' / 'data' / 'processed' / 'market_macro_features_daily.csv'

print('Project root:', ROOT)

In [ ]:
def load_daily_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=['date'])
    df['date'] = pd.to_datetime(df['date']).dt.normalize()
    return df.sort_values('date').drop_duplicates(subset=['date'], keep='last').reset_index(drop=True)


def log_return(series: pd.Series, periods: int = 1) -> pd.Series:
    return np.log(series / series.shift(periods))


def build_market_raw_with_gold() -> pd.DataFrame:
    market = load_daily_csv(MARKET_RAW_PATH)
    gold = load_daily_csv(GOLD_RAW_PATH)

    keep_market_cols = [col for col in market.columns if col not in {'gold_close', 'gold_source'}]
    market = market[keep_market_cols]

    gold = gold[['date', 'gold_close', 'gold_source']].copy()
    gold['gold_close'] = pd.to_numeric(gold['gold_close'], errors='coerce')

    merged = market.merge(gold, on='date', how='left')
    market_anchor_cols = ['spx_close', 'vix_close', 'dxy_close']
    merged = merged.loc[~merged[market_anchor_cols].isna().all(axis=1)].copy()
    merged = merged.sort_values('date').reset_index(drop=True)
    return merged


def build_features(market: pd.DataFrame, macro: pd.DataFrame) -> pd.DataFrame:
    macro = macro.copy()
    macro['real_yield_10y'] = pd.to_numeric(macro['real_yield_10y'], errors='coerce')

    daily = market.merge(macro[['date', 'real_yield_10y']], on='date', how='left')
    daily['real_yield_10y'] = daily['real_yield_10y'].ffill()
    daily = daily.sort_values('date').reset_index(drop=True)

    features = daily.copy()
    for col in ['spx_close', 'vix_close', 'dxy_close', 'gold_close']:
        features[col] = pd.to_numeric(features[col], errors='coerce')

    features['spx_ret_1d'] = log_return(features['spx_close'])
    features['gold_ret_1d'] = log_return(features['gold_close'])
    features['dxy_ret_1d'] = log_return(features['dxy_close'])

    features['vix_change_1d'] = features['vix_close'].diff(1)
    features['vix_change_5d'] = features['vix_close'].diff(5)
    features['real_yield_change_1d'] = features['real_yield_10y'].diff(1)
    features['real_yield_change_5d'] = features['real_yield_10y'].diff(5)

    features['spx_drawdown_1d'] = features['spx_ret_1d'].clip(upper=0)
    features['spx_drawdown_5d'] = log_return(features['spx_close'], periods=5).clip(upper=0)
    features['vix_ma_5d'] = features['vix_close'].rolling(5, min_periods=3).mean()
    features['gold_spx_corr_20d'] = features['gold_ret_1d'].rolling(20, min_periods=10).corr(features['spx_ret_1d'])

    return features

In [ ]:
market = build_market_raw_with_gold()
macro = load_daily_csv(MACRO_RAW_PATH)
features = build_features(market, macro)

MARKET_RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
FEATURES_PATH.parent.mkdir(parents=True, exist_ok=True)

market.to_csv(MARKET_RAW_PATH, index=False)
features.to_csv(FEATURES_PATH, index=False)

corr = features['gold_spx_corr_20d'].dropna()

print('Saved raw market file:', MARKET_RAW_PATH)
print('Saved features file:', FEATURES_PATH)
print('Rows:', len(features))
print('Date range:', features['date'].min().date(), 'to', features['date'].max().date())
print('Duplicate dates:', int(features['date'].duplicated().sum()))
print('Gold missing on market calendar:', int(features['gold_close'].isna().sum()))
print('Gold return non-null rows:', int(features['gold_ret_1d'].notna().sum()))
print('Gold/SPX corr non-null rows:', int(features['gold_spx_corr_20d'].notna().sum()))
if not corr.empty:
    print('Gold/SPX corr min/max:', float(corr.min()), float(corr.max()))

In [ ]:
from pathlib import Path
import shutil

ROOT = Path.cwd().resolve()
while not (ROOT / "plan_v2.md").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

INTEGRATION_DIR = ROOT / "data" / "integration"
INTEGRATION_DIR.mkdir(parents=True, exist_ok=True)

source_options = {
    "trends_features_daily.csv": [
        ROOT / "data" / "processed" / "trends_features_daily.csv",
        ROOT / "data" / "macro_trend" / "data" / "processed" / "trends_features_daily.csv",
    ],
    "market_macro_features_daily.csv": [
        ROOT / "data" / "processed" / "market_macro_features_daily.csv",
        ROOT / "data" / "macro_trend" / "data" / "processed" / "market_macro_features_daily.csv",
    ],
}

for filename, candidates in source_options.items():
    src = next((path for path in candidates if path.exists()), None)
    if src is None:
        raise FileNotFoundError(f"Could not find source file for {filename}: {candidates}")

    dst = INTEGRATION_DIR / filename
    shutil.copy2(src, dst)
    print(f"Copied {src} -> {dst}")


# Feature Integration for GMM and Stage 1

This notebook aligns and integrates the event-level, market-macro, and Google Trends features needed for the next modeling steps.

Outputs:
- `data/integration/daily_gmm_input_v1.csv`
- `data/integration/event_stage1_aligned_features_v1.csv`

Design rules:
- Market and macro features align to `feature_anchor_date`
- Google Trends aligns to `event_date_ny - 1 day` using backward lookup on the daily-available series
- Mixed timestamp parsing uses `format="mixed"` to avoid dropping valid rows


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "data" / "integration" / "stage1_event_level_modeling_input_v1_full.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/integration")


ROOT = find_project_root(Path.cwd().resolve())
INTEGRATION_DIR = ROOT / "data" / "integration"

MARKET_PATH = INTEGRATION_DIR / "market_macro_features_daily.csv"
TRENDS_PATH = INTEGRATION_DIR / "trends_features_daily.csv"
EVENT_PATH = INTEGRATION_DIR / "stage1_event_level_modeling_input_v1_full.csv"

DAILY_OUT = INTEGRATION_DIR / "daily_gmm_input_v1.csv"
EVENT_OUT = INTEGRATION_DIR / "event_stage1_aligned_features_v1.csv"

print(f"ROOT: {ROOT}")
print(f"MARKET_PATH: {MARKET_PATH}")
print(f"TRENDS_PATH: {TRENDS_PATH}")
print(f"EVENT_PATH: {EVENT_PATH}")


## Load Daily Inputs

The market-macro table is already on a trading-day calendar. The Trends table is a daily-available natural-day series. We keep both date columns as plain calendar dates for merging.


In [ ]:
market = pd.read_csv(MARKET_PATH)
trends = pd.read_csv(TRENDS_PATH)

market["date"] = pd.to_datetime(market["date"]).dt.normalize()
trends["date"] = pd.to_datetime(trends["date"]).dt.normalize()

print("market shape:", market.shape)
print("trends shape:", trends.shape)
display(market.head())
display(trends.head())


## Build Trading-Day Daily Table

For GMM and any daily downstream work, we use the market table as the master trading-day calendar and left-join Trends on `date`.


In [ ]:
daily_gmm_input = market.merge(trends, on="date", how="left", validate="one_to_one")
daily_gmm_input.to_csv(DAILY_OUT, index=False)

print("daily_gmm_input shape:", daily_gmm_input.shape)
print("saved to:", DAILY_OUT)
display(daily_gmm_input.head())
display(daily_gmm_input.tail())


## Load Event Table and Parse Mixed Timestamps

A subset of event timestamps uses mixed datetime formats. Using `format="mixed"` prevents valid second-term rows from being lost during parsing.


In [ ]:
events = pd.read_csv(EVENT_PATH)

events["event_time_ny_ts"] = pd.to_datetime(
    events["event_time_ny"],
    errors="coerce",
    utc=True,
    format="mixed",
)
events["event_time_utc_ts"] = pd.to_datetime(
    events["event_time_utc"],
    errors="coerce",
    utc=True,
    format="mixed",
)

events["feature_anchor_date_dt"] = pd.to_datetime(
    events["feature_anchor_date"],
    errors="coerce",
    utc=True,
    format="mixed",
).dt.tz_convert(None).dt.normalize()

events["target_trade_date_dt"] = pd.to_datetime(
    events["target_trade_date"],
    errors="coerce",
    utc=True,
    format="mixed",
).dt.tz_convert(None).dt.normalize()

events["event_date_ny"] = events["event_time_ny_ts"].dt.tz_convert("America/New_York").dt.normalize().dt.tz_localize(None)
events["trend_lookup_date"] = events["event_date_ny"] - pd.Timedelta(days=1)

print("events shape:", events.shape)
print("missing parsed event_time_ny_ts:", int(events["event_time_ny_ts"].isna().sum()))
print("missing parsed event_time_utc_ts:", int(events["event_time_utc_ts"].isna().sum()))
display(events[["event_id", "event_time_ny", "event_time_ny_ts", "feature_anchor_date", "feature_anchor_date_dt", "trend_lookup_date"]].head())


## Attach Market-Macro Features to Events

Market and macro features use the pre-defined `feature_anchor_date`, which already follows the project leakage controls.


In [ ]:
market_for_events = market.rename(columns={"date": "market_feature_date"})

event_with_market = events.merge(
    market_for_events,
    left_on="feature_anchor_date_dt",
    right_on="market_feature_date",
    how="left",
    validate="many_to_one",
)

print("event_with_market shape:", event_with_market.shape)
print("missing market_feature_date:", int(event_with_market["market_feature_date"].isna().sum()))
display(event_with_market[["event_id", "feature_anchor_date_dt", "market_feature_date", "spx_close", "gold_close", "real_yield_10y"]].head())


## Attach Google Trends Features to Events

Trends should reflect only information available before the event. We therefore use `trend_lookup_date = event_date_ny - 1 day` and merge backward on the daily-available series.


In [ ]:
trend_feature_cols = [c for c in trends.columns if c != "date"]
trends_for_events = trends.rename(columns={"date": "trends_feature_date"}).sort_values("trends_feature_date")

event_with_market = event_with_market.sort_values("trend_lookup_date").reset_index(drop=True)

event_stage1_aligned = pd.merge_asof(
    event_with_market,
    trends_for_events,
    left_on="trend_lookup_date",
    right_on="trends_feature_date",
    direction="backward",
)

event_stage1_aligned = event_stage1_aligned.sort_values(["event_time_utc_ts", "event_id"]).reset_index(drop=True)
event_stage1_aligned.to_csv(EVENT_OUT, index=False)

print("event_stage1_aligned shape:", event_stage1_aligned.shape)
print("saved to:", EVENT_OUT)
display(event_stage1_aligned[["event_id", "event_date_ny", "trend_lookup_date", "trends_feature_date", "trend_tariff_z_60d", "trend_inflation_z_60d", "trend_war_z_60d"]].head())


## Quality Checks

These checks help confirm that:
- the daily table stays on the trading-day calendar
- all event rows survive the alignment
- missingness is limited to expected warm-up periods or true data gaps


In [ ]:
daily_qc = {
    "daily_rows": len(daily_gmm_input),
    "daily_date_min": str(daily_gmm_input["date"].min().date()),
    "daily_date_max": str(daily_gmm_input["date"].max().date()),
    "daily_unique_dates": int(daily_gmm_input["date"].nunique()),
    "daily_missing_trend_tariff_z_60d": int(daily_gmm_input["trend_tariff_z_60d"].isna().sum()),
    "daily_missing_trend_inflation_z_60d": int(daily_gmm_input["trend_inflation_z_60d"].isna().sum()),
    "daily_missing_trend_war_z_60d": int(daily_gmm_input["trend_war_z_60d"].isna().sum()),
}

event_qc = {
    "event_rows": len(event_stage1_aligned),
    "event_unique_ids": int(event_stage1_aligned["event_id"].nunique()),
    "missing_event_time_ny_ts": int(event_stage1_aligned["event_time_ny_ts"].isna().sum()),
    "missing_market_feature_date": int(event_stage1_aligned["market_feature_date"].isna().sum()),
    "missing_trends_feature_date": int(event_stage1_aligned["trends_feature_date"].isna().sum()),
    "missing_trend_tariff_z_60d": int(event_stage1_aligned["trend_tariff_z_60d"].isna().sum()),
    "missing_trend_inflation_z_60d": int(event_stage1_aligned["trend_inflation_z_60d"].isna().sum()),
    "missing_trend_war_z_60d": int(event_stage1_aligned["trend_war_z_60d"].isna().sum()),
}

print("daily_qc")
display(pd.DataFrame([daily_qc]))
print("event_qc")
display(pd.DataFrame([event_qc]))


## Suggested Next Use

- `daily_gmm_input_v1.csv`: use for GMM regime fitting and daily diagnostics
- `event_stage1_aligned_features_v1.csv`: use for Stage 1 event-level modeling and event-to-regime joins
